# 🌡️ Temperature Sensor Fault Detection
## Phase 3: Feature Extraction

---
**Goal:** Convert each 100-sample window into a small set of meaningful numbers (features)  
**Input:** `temp_labeled_dataset.parquet` from Phase 2 (windows + labels)  
**Output:** `temp_features.parquet` — one row per window, ready for ML training

### Why Feature Extraction?
Instead of feeding 100 raw temperature values into a model,  
we compute **summary statistics** that capture what makes each fault class unique:

| Feature Group | Examples | Catches |
|---|---|---|
| Central tendency | mean, median | Bias/drift (mean shifts) |
| Spread | std, variance, range | Noise faults (high spread) |
| Shape | skewness, kurtosis | Spike asymmetry |
| Flatness | std ≈ 0, unique count | Flat-line faults |
| Rate of change | mean abs diff, max diff | Spikes (sudden jumps) |
| Crossing | zero crossing rate | Signal activity level |

### Sections
1. Imports & Load Phase 2 Data  
2. Feature Engineering Functions  
3. Extract Features from All Windows  
4. Feature Overview & Statistics  
5. Feature Distributions per Class  
6. Feature Correlation & Redundancy  
7. Most Discriminative Features  
8. Train/Test Split  
9. Save for Phase 4  

---
## 1. Imports & Load Phase 2 Data

In [ ]:
import os, json, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy          import stats
from scipy.stats    import skew, kurtosis
from scipy.signal   import welch
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

CLASS_COLORS = {
    'normal'        : '#43A047',
    'noise_fault'   : '#E53935',
    'bias_fault'    : '#FB8C00',
    'flatline_fault': '#8E24AA',
}

print('✅ Imports done.')

In [ ]:
# ================================================================
#  ONLY CHANGE THIS if your path is different
# ================================================================
BASE_PATH  = r'C:\PVS_Dataset'
PHASE2_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase2')
OUTPUT_DIR = os.path.join(BASE_PATH, 'outputs_temp_phase3')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load Phase 2 config
with open(os.path.join(PHASE2_DIR, 'temp_phase2_config.json')) as f:
    cfg = json.load(f)

WINDOW_SIZE  = cfg['WINDOW_SIZE']
SAMPLING_HZ  = cfg['SAMPLING_HZ']
TEMP_COL     = cfg['TEMP_COL']
GLOBAL_MEAN  = cfg['GLOBAL_MEAN']
GLOBAL_STD   = cfg['GLOBAL_STD']
window_cols  = [f'temp_{i}' for i in range(WINDOW_SIZE)]
CLASS_LABELS = {int(k):v for k,v in cfg['CLASS_LABELS'].items()}

print('✅ Phase 2 config loaded.')
print(f'   Window size  : {WINDOW_SIZE} samples')
print(f'   Sampling Hz  : {SAMPLING_HZ}')
print(f'   Temp column  : {TEMP_COL}')

In [ ]:
# Load labeled dataset from Phase 2
dataset_path = os.path.join(PHASE2_DIR, 'temp_labeled_dataset.parquet')
labeled_df   = pd.read_parquet(dataset_path)

print(f'✅ Labeled dataset loaded: {labeled_df.shape}')
print(f'   Total windows : {len(labeled_df):,}')
print(f'   RAM           : {labeled_df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print()
print('Class distribution:')
vc = labeled_df['label_name'].value_counts()
for name, cnt in vc.items():
    print(f'  {name:20s}: {cnt:>7,}  ({cnt/len(labeled_df)*100:.1f}%)')

---
## 2. Feature Engineering Functions

We compute **13 features** per window — each one chosen because it reveals something  
specific about at least one fault type.

In [ ]:
FEATURE_INFO = {
    'mean'          : 'Average value — shifts up/down in bias fault',
    'std'           : 'Standard deviation — near 0 in flatline, high in noise',
    'variance'      : 'Spread squared — amplifies std differences',
    'rms'           : 'Root mean square — energy of signal',
    'min'           : 'Minimum value — negative spikes pull this down',
    'max'           : 'Maximum value — positive spikes pull this up',
    'range'         : 'max - min — zero in flatline, huge in noise',
    'skewness'      : 'Asymmetry — noise spikes create asymmetry',
    'kurtosis'      : 'Tail heaviness — spike faults have very high kurtosis',
    'mean_abs_diff' : 'Average step size between samples — huge in noise, 0 in flatline',
    'max_diff'      : 'Largest single step — directly catches spike moment',
    'trend'         : 'Linear slope — positive/negative in bias fault, 0 in normal/flatline',
    'unique_ratio'  : 'Unique values / window size — ~0 in flatline (all same value)',
}

print('📋 Features to be extracted (13 total):')
print(f'{"Feature":<18} {"Purpose"}')
print('-' * 70)
for feat, desc in FEATURE_INFO.items():
    print(f'  {feat:<16} {desc}')

In [ ]:
def extract_features(window: np.ndarray) -> dict:
    """
    Extract 13 features from a single temperature window.
    Input : numpy array of shape (WINDOW_SIZE,)
    Output: dict of feature_name -> float value
    """
    w = window.astype(np.float64)
    n = len(w)

    # ── Central tendency ──
    feat_mean     = np.mean(w)

    # ── Spread ──
    feat_std      = np.std(w)
    feat_var      = np.var(w)
    feat_rms      = np.sqrt(np.mean(w**2))

    # ── Extremes ──
    feat_min      = np.min(w)
    feat_max      = np.max(w)
    feat_range    = feat_max - feat_min

    # ── Shape ──
    feat_skew     = float(skew(w))     if feat_std > 1e-10 else 0.0
    feat_kurt     = float(kurtosis(w)) if feat_std > 1e-10 else 0.0

    # ── Rate of change ──
    diffs             = np.abs(np.diff(w))
    feat_mean_diff    = np.mean(diffs)
    feat_max_diff     = np.max(diffs)

    # ── Trend (linear slope via polyfit) ──
    x             = np.arange(n)
    feat_trend    = float(np.polyfit(x, w, 1)[0])   # slope of best-fit line

    # ── Flatness indicator ──
    feat_unique   = len(np.unique(np.round(w, 4))) / n   # near 0 → flatline

    return {
        'mean'         : feat_mean,
        'std'          : feat_std,
        'variance'     : feat_var,
        'rms'          : feat_rms,
        'min'          : feat_min,
        'max'          : feat_max,
        'range'        : feat_range,
        'skewness'     : feat_skew,
        'kurtosis'     : feat_kurt,
        'mean_abs_diff': feat_mean_diff,
        'max_diff'     : feat_max_diff,
        'trend'        : feat_trend,
        'unique_ratio' : feat_unique,
    }


# ── Quick sanity check ──
test_normal   = np.full(WINDOW_SIZE, GLOBAL_MEAN, dtype=np.float64) + np.random.normal(0, max(GLOBAL_STD,0.01), WINDOW_SIZE)
test_flatline = np.full(WINDOW_SIZE, GLOBAL_MEAN, dtype=np.float64)
test_spike    = test_normal.copy(); test_spike[[10,50,90]] += max(GLOBAL_STD*8, 0.5)

print('Sanity check — 3 synthetic windows:')
print(f'{"Feature":<18} {"Normal":>12} {"Flatline":>12} {"Spike":>12}')
print('-'*56)
fn = extract_features(test_normal)
ff = extract_features(test_flatline)
fs = extract_features(test_spike)
for k in fn:
    print(f'  {k:<16} {fn[k]:>12.6f} {ff[k]:>12.6f} {fs[k]:>12.6f}')

print('\n✅ Feature function works correctly.')

---
## 3. Extract Features from All Windows

Apply `extract_features()` to every row in the labeled dataset.  
This converts the 100-column window matrix into a 13-column feature matrix.

In [ ]:
import time
print(f'Extracting features from {len(labeled_df):,} windows...')
t0 = time.time()

# Get raw window values as numpy matrix — faster than row-by-row
windows_matrix = labeled_df[window_cols].values  # shape: (N, 100)

# Extract features for every row
feature_list = []
for i in range(len(windows_matrix)):
    feature_list.append(extract_features(windows_matrix[i]))
    if (i+1) % 50_000 == 0:
        print(f'  Processed {i+1:,} / {len(windows_matrix):,}  ({(i+1)/len(windows_matrix)*100:.0f}%)')

features_df = pd.DataFrame(feature_list)

elapsed = time.time() - t0
print(f'\n✅ Feature extraction done in {elapsed:.1f}s')
print(f'   Features shape : {features_df.shape}')
print(f'   Features       : {list(features_df.columns)}')

In [ ]:
# Combine features with metadata + labels
META_COLS = ['label', 'label_name', 'dataset_id', 'side', 'vehicle', 'driver', 'scenario']
FEAT_COLS = list(features_df.columns)

full_df = pd.concat([
    labeled_df[META_COLS].reset_index(drop=True),
    features_df.reset_index(drop=True)
], axis=1)

print('Combined feature DataFrame:')
print(f'  Shape   : {full_df.shape}')
print(f'  Columns : {META_COLS} + {FEAT_COLS}')
display(full_df.head(5))

---
## 4. Feature Overview & Statistics

In [ ]:
print('📊 FEATURE STATISTICS — OVERALL')
display(full_df[FEAT_COLS].describe().round(6))

In [ ]:
print('📊 FEATURE MEANS PER CLASS')
per_class = full_df.groupby('label_name')[FEAT_COLS].mean().round(6)
display(per_class)

print('\n📌 Key differences visible in the table:')
print('  std / range / mean_abs_diff / max_diff  → highest in noise_fault, ~0 in flatline')
print('  trend                                   → non-zero in bias_fault')
print('  unique_ratio                            → near 0 in flatline_fault')
print('  kurtosis                                → high in noise_fault (heavy tails)')

---
## 5. Feature Distributions per Class

For each feature, we plot its distribution for all 4 classes.  
A **good feature** has clearly separated distributions — the model can easily split the classes.  
A **bad feature** has overlapping distributions — not useful for classification.

In [ ]:
n_feats = len(FEAT_COLS)
ncols   = 3
nrows   = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5*nrows))
axes = axes.flatten()

for i, feat in enumerate(FEAT_COLS):
    for cls_name, color in CLASS_COLORS.items():
        data = full_df[full_df['label_name']==cls_name][feat].dropna()
        # Clip extreme outliers for plotting clarity
        p1, p99 = np.percentile(data, [1, 99])
        data_clipped = data.clip(p1, p99)
        axes[i].hist(data_clipped, bins=60, alpha=0.45, color=color,
                      density=True, label=cls_name)

    axes[i].set_title(f'{feat}\n→ {FEATURE_INFO[feat][:45]}',
                       fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=7)
    axes[i].grid(True, alpha=0.3)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions per Class\n(separated distributions = better feature)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'feat_01_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: feat_01_distributions.png')

In [ ]:
# Box plots: feature value per class — great for spotting separability
KEY_FEATS = ['std', 'range', 'kurtosis', 'mean_abs_diff', 'max_diff', 'trend', 'unique_ratio']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

class_order = ['normal', 'noise_fault', 'bias_fault', 'flatline_fault']

for i, feat in enumerate(KEY_FEATS):
    data_per_class = [full_df[full_df['label_name']==c][feat].clip(
                          *np.percentile(full_df[feat],[0.5,99.5])).values
                      for c in class_order]
    bp = axes[i].boxplot(data_per_class, patch_artist=True, labels=class_order,
                          medianprops=dict(color='black', linewidth=2),
                          flierprops=dict(marker='o', markersize=1.5, alpha=0.2))
    for patch, cls in zip(bp['boxes'], class_order):
        patch.set_facecolor(CLASS_COLORS[cls])
        patch.set_alpha(0.6)
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20, labelsize=8)
    axes[i].grid(True, alpha=0.3, axis='y')

axes[-1].set_visible(False)
fig.suptitle('Key Feature Separability by Class\n(well-separated boxes = useful feature)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'feat_02_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: feat_02_boxplots.png')

---
## 6. Feature Correlation & Redundancy

Highly correlated features carry the same information.  
We identify and optionally remove redundant ones.

In [ ]:
corr = full_df[FEAT_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
             cmap='RdYlGn', center=0, vmin=-1, vmax=1,
             linewidths=0.4, annot_kws={'size':8}, ax=ax,
             cbar_kws={'label':'Pearson r'})
ax.set_title('Feature Correlation Matrix\n(high |r| = redundant pair, consider removing one)',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'feat_03_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: feat_03_correlation.png')

# Print highly correlated pairs
print('\nHighly correlated feature pairs (|r| > 0.85):')
found = False
for i in range(len(FEAT_COLS)):
    for j in range(i+1, len(FEAT_COLS)):
        r = corr.iloc[i,j]
        if abs(r) > 0.85:
            print(f'  {FEAT_COLS[i]:18s} ↔ {FEAT_COLS[j]:18s}  r = {r:.3f}')
            found = True
if not found:
    print('  None found — all features contribute unique information. Keep all 13.')

In [ ]:
# Auto-remove one from each highly correlated pair (|r| > 0.90)
# We keep the feature that appears first in FEAT_COLS
THRESHOLD = 0.90
to_drop   = set()

for i in range(len(FEAT_COLS)):
    for j in range(i+1, len(FEAT_COLS)):
        if abs(corr.iloc[i,j]) > THRESHOLD and FEAT_COLS[j] not in to_drop:
            to_drop.add(FEAT_COLS[j])

FINAL_FEATS = [f for f in FEAT_COLS if f not in to_drop]

if to_drop:
    print(f'Dropped {len(to_drop)} redundant features: {to_drop}')
else:
    print('No features dropped — all are non-redundant.')

print(f'\nFinal feature set ({len(FINAL_FEATS)} features):')
for f in FINAL_FEATS:
    print(f'  {f}')

---
## 7. Most Discriminative Features

**Mutual Information** measures how much each feature tells us about the class label.  
Higher = more useful for classification.

In [ ]:
X_mi = full_df[FINAL_FEATS].fillna(0).values
y_mi = full_df['label'].values

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_df = pd.DataFrame({'Feature': FINAL_FEATS, 'Mutual Info': mi_scores})
mi_df = mi_df.sort_values('Mutual Info', ascending=False).reset_index(drop=True)

print('📊 MUTUAL INFORMATION SCORES (higher = more useful for classification)')
print('='*50)
for _, row in mi_df.iterrows():
    bar = '█' * int(row['Mutual Info'] * 60)
    print(f'  {row["Feature"]:18s}  {row["Mutual Info"]:.4f}  {bar}')

fig, ax = plt.subplots(figsize=(11, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(mi_df))[::-1])
bars = ax.barh(mi_df['Feature'], mi_df['Mutual Info'],
                color=colors, edgecolor='black', linewidth=0.5)
for bar_i, val in zip(bars, mi_df['Mutual Info']):
    ax.text(val + 0.003, bar_i.get_y() + bar_i.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9)
ax.set_title('Feature Importance — Mutual Information Score\n(how much each feature predicts the fault class)',
              fontsize=12, fontweight='bold')
ax.set_xlabel('Mutual Information')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'feat_04_mutual_info.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: feat_04_mutual_info.png')

In [ ]:
# ── 2D scatter: top 2 features colored by class ──
# Shows how well the top features separate the 4 classes visually
top2 = mi_df['Feature'].iloc[:2].tolist()
top3 = mi_df['Feature'].iloc[:3].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sample = full_df.sample(min(8000, len(full_df)), random_state=42)

for ax, (fx, fy) in zip(axes, [(top2[0], top2[1]), (top3[0], top3[2])]):
    for cls, color in CLASS_COLORS.items():
        sub = sample[sample['label_name']==cls]
        xd  = sub[fx].clip(*np.percentile(sub[fx],[0.5,99.5]))
        yd  = sub[fy].clip(*np.percentile(sub[fy],[0.5,99.5]))
        ax.scatter(xd, yd, c=color, alpha=0.25, s=8, label=cls)
    ax.set_xlabel(fx, fontsize=10)
    ax.set_ylabel(fy, fontsize=10)
    ax.set_title(f'{fx}  vs  {fy}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, markerscale=2)
    ax.grid(True, alpha=0.3)

fig.suptitle('Top Features — 2D Class Separability\n(well-clustered colors = good feature pair)',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'feat_05_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📁 Saved: feat_05_scatter.png')

---
## 8. Train / Test Split

We split 80% train / 20% test, **stratified by class** so every class has equal representation in both sets.  
We also apply **StandardScaler** to normalize features — required for SVM and KNN.

In [ ]:
X = full_df[FINAL_FEATS].fillna(0).values
y = full_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train / Test Split (80/20 stratified):')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')

print('\nClass distribution in train set:')
for lbl, name in CLASS_LABELS.items():
    cnt = (y_train==lbl).sum()
    print(f'  Class {lbl} ({name:15s}): {cnt:>7,}  ({cnt/len(y_train)*100:.1f}%)')

print('\nClass distribution in test set:')
for lbl, name in CLASS_LABELS.items():
    cnt = (y_test==lbl).sum()
    print(f'  Class {lbl} ({name:15s}): {cnt:>7,}  ({cnt/len(y_test)*100:.1f}%)')

In [ ]:
# Fit scaler on TRAIN only — never fit on test data
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('✅ StandardScaler applied:')
print(f'   Train mean (after scaling): {X_train_s.mean():.6f}  (should be ≈0)')
print(f'   Train std  (after scaling): {X_train_s.std():.6f}   (should be ≈1)')
print(f'   Test  mean (after scaling): {X_test_s.mean():.4f}')
print()
print('Note: X_train / X_test = raw (for tree-based models like Random Forest, XGBoost)')
print('      X_train_s / X_test_s = scaled (for SVM, KNN)')

---
## 9. Save for Phase 4

In [ ]:
import pickle

# Save full feature DataFrame
feat_path = os.path.join(OUTPUT_DIR, 'temp_features.parquet')
full_df.to_parquet(feat_path, index=False)
print(f'✅ Feature dataset saved  → {feat_path}')
print(f'   Size: {os.path.getsize(feat_path)/1e6:.2f} MB  |  Rows: {len(full_df):,}  |  Features: {len(FINAL_FEATS)}')

# Save train/test splits as numpy arrays
splits = {
    'X_train'   : X_train,
    'X_test'    : X_test,
    'X_train_s' : X_train_s,
    'X_test_s'  : X_test_s,
    'y_train'   : y_train,
    'y_test'    : y_test,
}
splits_path = os.path.join(OUTPUT_DIR, 'temp_splits.pkl')
with open(splits_path, 'wb') as f:
    pickle.dump(splits, f)
print(f'✅ Train/test splits saved → {splits_path}')

# Save scaler
scaler_path = os.path.join(OUTPUT_DIR, 'temp_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'✅ Scaler saved           → {scaler_path}')

# Save Phase 3 config
p3_cfg = {
    'TEMP_COL'      : TEMP_COL,
    'FINAL_FEATS'   : FINAL_FEATS,
    'ALL_FEATS'     : FEAT_COLS,
    'DROPPED_FEATS' : list(to_drop),
    'CLASS_LABELS'  : CLASS_LABELS,
    'WINDOW_SIZE'   : WINDOW_SIZE,
    'SAMPLING_HZ'   : SAMPLING_HZ,
    'N_TRAIN'       : int(len(X_train)),
    'N_TEST'        : int(len(X_test)),
    'mi_ranking'    : mi_df['Feature'].tolist(),
    'BASE_PATH'     : BASE_PATH,
    'OUTPUT_DIR'    : OUTPUT_DIR,
    'feat_path'     : feat_path,
    'splits_path'   : splits_path,
    'scaler_path'   : scaler_path,
}
cfg_path = os.path.join(OUTPUT_DIR, 'temp_phase3_config.json')
with open(cfg_path, 'w') as f:
    json.dump(p3_cfg, f, indent=2)
print(f'✅ Phase 3 config saved   → {cfg_path}')

# Final summary
print('\n' + '='*60)
print('PHASE 3 COMPLETE — FEATURE EXTRACTION')
print('='*60)
print(f'  Total features extracted : {len(FEAT_COLS)}')
print(f'  Features after pruning   : {len(FINAL_FEATS)}')
print(f'  Features dropped         : {len(to_drop)}')
print(f'  Training samples         : {len(X_train):,}')
print(f'  Test samples             : {len(X_test):,}')
print(f'  Top feature (MI score)   : {mi_df["Feature"].iloc[0]}  ({mi_df["Mutual Info"].iloc[0]:.4f})')
print()
print('  Saved files:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f'    {fname:40s} {sz:6.1f} KB')
print(f'\n  ➡️  Open Phase4_Model_Training.ipynb to continue.')